In [8]:
import P04_Parser
from igor2 import binarywave
import numpy as np
import matplotlib.pyplot as plt
import re
import hyperspy.api as hs
import matplotlib as mpl
import matplotlib.pyplot as plt
from pathlib import Path

def load_ibw_swept_as_kmap(file):
    """Swept (2D) .ibw cut -> Kmap with 3 identical theta_y slices, so P04_Parser plotting works."""
    wave = binarywave.load(file)["wave"]
    data = np.array(wave["wData"], dtype=float)     # (energy, angle)
    sfa = wave["wave_header"]["sfA"]
    sfb = wave["wave_header"]["sfB"]
    energy = sfb[0] + sfa[0] * np.arange(data.shape[0])
    theta = sfb[1] + sfa[1] * np.arange(data.shape[1])
    data_map = np.repeat(data[:, :, np.newaxis], 3, axis=2)
    return P04_Parser.Kmap(data_map, energy, theta, np.array([-1, 0, 1]))


In [9]:
kmap_LV_47ev = P04_Parser.Kmap.from_file(r"T:\Beamtimes\MaxIV\Oct_26\raw\FeSi_2A_day1\FeSi_2A_kmap_align_hv_47_LV_0001.zip")

kmap_LV_47ev.convert_to_k_space()


In [38]:
%matplotlib qt
from P04_Parser.Overlay import RectangleBZ

spans, sigs = kmap_LV_47ev.plot("all", cmap ="viridis", return_sig=True)

a = 4.49

Bz = RectangleBZ(a_0=a/np.sqrt(1) , b_0=a/np.sqrt(2), repeat_y=[-1,1], repeat_x=[-1,1], rotation=90,)

Bz.add_to_plot(sigs[0], mark_high_symmetry=True, labels= ['Γ',"X","M","R"])

In [23]:
hvscan_LV_GX = P04_Parser.HvScan.from_file(r"T:\Beamtimes\MaxIV\Oct_26\raw\FeSi_2A_day1\FeSi_2A_hvscan_18_120_LV_GX_0001hv scan.ibw")


In [34]:
hvscan_LV_GX.plot("all", cmap="viridis", return_sig=False)



(SpanROI(left=-1.4678, right=-1.17424),
 SpanROI(left=-16.3384, right=-9.7082),
 SpanROI(left=18, right=38.4))

In [ ]:
from P04_Parser.Overlay import RectangleBZ, PhotonEnergyLines

inner_potential = 10  # eV, the default convert_to_k_space uses; PhotonEnergyLines must use the same value
hvscan_LV_GX_k = hvscan_LV_GX.convert_to_k_space(inner_potential=inner_potential)


In [35]:
%matplotlib qt

spans, sigs = hvscan_LV_GX_k.plot("all", cmap="viridis", return_sig=True)

a = 4.49
b = a / np.sqrt(2)  # spacing along k_z, same as the kmap overlay above
G_z = 2 * np.pi / b

# Put the BZ origin on the Γ point closest to the middle of the measured k_z range
# (after k-conversion, photon_energy holds the k_z axis)
kz_center = (hvscan_LV_GX_k.photon_energy.min() + hvscan_LV_GX_k.photon_energy.max()) / 2
origin = [0, round(kz_center / G_z) * G_z]

Bz = RectangleBZ(a_0=a, b_0=b, origin=origin, repeat_x=[-1, 1], repeat_y=[-1, 1])#, rotation = 90)
Bz.add_to_plot(sigs[0], mark_high_symmetry=True, labels=["Γ", "X", "M", "R"])

hv_lines = PhotonEnergyLines(
    hvs=np.arange(20, 121, 10),
    inner_potential=inner_potential,
    angle_range=[hvscan_LV_GX._theta[0], hvscan_LV_GX._theta[-1]],
)
hv_lines.add_to_plot(sigs[0])